# Tier 2 - rozproszony magazyn artefaktów (Ray HDFS)

**Autor:** kaliński_jakub_420841  
**Punktacja (ray-zadanie.pdf):** 6–7 pkt baza + do +2 pkt bonus Actor API + 1 pkt klaster rozproszony.

## Architektura

- **NameNode** (aktor) - wyłącznie **metadane** artefaktów i bloków; nie przechowuje treści pliku.
- **DataNode** (aktor × N) - przechowuje **chunki/bloki** (`bytes`).
- **ArtifactClient** (driver) - planuje operacje u NameNode, dane zapisuje **bezpośrednio** na DataNodes (jak w HDFS).

DataNodes **nie uzgadniają** między sobą - placement i replikację decyduje NameNode.

```mermaid
flowchart LR
  Client --> NameNode
  Client --> DataNode1
  Client --> DataNode2
  NameNode -.->|replikacja po awarii| DataNode2
```



### Środowisko

- **Docker (zalecane):** `docker compose up -d` lub `docker compose -f docker-compose.tier2.yaml up -d`
- **Lokalnie w kontenerze:** `USE_REMOTE_CLUSTER = False`
- **Klaster head+worker (+1 pkt):** compose tier2 ustawia `RAY_ADDRESS=ray://ray-head:10001` - notebook wykrywa to automatycznie.



In [1]:
import os
import sys
import json
from pathlib import Path

import ray

# Moduł hdfs: katalog for-students/hdfs (Docker: /home/ray/hdfs)
HDFS_ROOT = None
for root in [Path.cwd().parent, Path("/home/ray")]:
    if (root / "hdfs" / "__init__.py").is_file():
        HDFS_ROOT = root
        sys.path.insert(0, str(root))
        break
if HDFS_ROOT is None:
    raise RuntimeError("Nie znaleziono pakietu hdfs — uruchom w Dockerze z mountem ./hdfs")

from hdfs import BLOCK_SIZE, DEFAULT_REPLICATION_FACTOR, ArtifactClient
from hdfs.data_node import DataNode
from hdfs.name_node import NameNode

print("Ray:", ray.__version__)
print("BLOCK_SIZE:", BLOCK_SIZE, "| domyślna replikacja:", DEFAULT_REPLICATION_FACTOR)



Ray: 2.55.1
BLOCK_SIZE: 256 | domyślna replikacja: 2


In [2]:
STUDENT_ID = "kaliński_jakub_420841"
NUM_DATA_NODES = 3

# False = lokalny Ray w kontenerze / na maszynie. True = Ray Client (compose tier2).
USE_REMOTE_CLUSTER = os.environ.get("RAY_ADDRESS", "").startswith("ray://")
RAY_ADDRESS = os.environ.get("RAY_ADDRESS", "ray://ray-head:10001")

if ray.is_initialized():
    ray.shutdown()

if USE_REMOTE_CLUSTER:
    # Workery klastra muszą widzieć pakiet hdfs (mount ./hdfs na head/worker w compose).
    ray.init(
        address=RAY_ADDRESS,
        ignore_reinit_error=True,
        namespace=f"tier2-hdfs-{STUDENT_ID}",
        runtime_env={
            "env_vars": {"PYTHONPATH": str(HDFS_ROOT)},
            "working_dir": str(HDFS_ROOT),
            "excludes": [
                "work",
                "work/**",
                "**/.ipynb_checkpoints/**",
                "**/*.ipynb",
            ],
        },
    )
    mode = "remote (Ray Client - klaster rozproszony)"
else:
    ray.init(
        ignore_reinit_error=True,
        namespace=f"tier2-hdfs-{STUDENT_ID}",
    )
    mode = "local"

print("Połączono - tryb:", mode)
print("Namespace:", ray.get_runtime_context().namespace)
print("Zasoby:", ray.cluster_resources())



2026-06-01 02:52:03,340	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 2147483648 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.42gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-06-01 02:52:08,481	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Połączono - tryb: local
Namespace: tier2-hdfs-kaliński_jakub_420841
Zasoby: {'CPU': 8.0, 'node:172.18.0.2': 1.0, 'memory': 5520122676.0, 'node:__internal_head__': 1.0, 'object_store_memory': 2365766860.0}


/home/ray/anaconda3/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Inicjalizacja klastra aktorów

Tworzymy 3 DataNodes (`SPREAD` - rozłożenie po węzłach Ray) i 1 NameNode.  
NameNode rejestruje id węzłów i handle do wywołań replikacji przy awarii.



In [3]:
def bootstrap_cluster(num_data_nodes=NUM_DATA_NODES):
    """Uruchom aktorów storage + metadata i zwróć klienta."""
    data_handles = {}
    for i in range(num_data_nodes):
        node_id = f"dn-{i}"
        # DataNode: @ray.remote(..., scheduling_strategy="SPREAD", max_restarts=1)
        data_handles[node_id] = DataNode.remote(node_id)

    name_node = NameNode.remote(block_size=BLOCK_SIZE)
    ray.get(name_node.register_data_nodes.remote(list(data_handles.keys())))
    for node_id, handle in data_handles.items():
        ray.get(name_node.set_data_node_handle.remote(node_id, handle))

    client = ArtifactClient(name_node, data_handles, block_size=BLOCK_SIZE)
    return name_node, data_handles, client


name_node, data_handles, client = bootstrap_cluster()
print("DataNodes:", list(data_handles.keys()))



DataNodes: ['dn-0', 'dn-1', 'dn-2']


## Test 1 - create + get

Tworzymy artefakt `demo.txt` (kilka bloków przy `BLOCK_SIZE=256`), replikacja **2**.  
Przepływ: `plan_create` -> klient `put_block` na wskazane węzły -> `commit_create`.



In [4]:
ARTIFACT = "demo.txt"
CONTENT = "Lorem ipsum " * 80  # ~960 B => kilka bloków

client.create(ARTIFACT, CONTENT, replication_factor=2)
retrieved = client.get(ARTIFACT)
assert retrieved == CONTENT, "create/get: treść się nie zgadza"

print("OK - create/get, długość:", len(retrieved))
print(json.dumps(client.list_cluster_state(), indent=2, ensure_ascii=False))



OK - create/get, długość: 960
{
  "artifacts": [
    {
      "name": "demo.txt",
      "size": 960,
      "blocks": 4,
      "replication_factor": 2
    }
  ],
  "data_nodes": [
    {
      "node_id": "dn-0",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-0",
        "status": "ALIVE",
        "block_count": 3
      }
    },
    {
      "node_id": "dn-1",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-1",
        "status": "ALIVE",
        "block_count": 3
      }
    },
    {
      "node_id": "dn-2",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-2",
        "status": "ALIVE",
        "block_count": 2
      }
    }
  ]
}


## Test 2 - listowanie stanu węzłów

Po każdej operacji weryfikujemy stan przez `list_artifacts` (NameNode) i `list_blocks` (DataNodes).



In [5]:
print("=== Artefakty (NameNode) ===")
print(json.dumps(ray.get(name_node.list_artifacts.remote()), indent=2, ensure_ascii=False))

print("\n=== Bloki na DataNodes ===")
for node_id, blocks in client.list_all_blocks_on_nodes().items():
    n = len(blocks) if isinstance(blocks, list) else 0
    print(f"  {node_id}: {n} bloków")
    if isinstance(blocks, list) and blocks:
        print("    przykład:", blocks[:2])



=== Artefakty (NameNode) ===
[
  {
    "name": "demo.txt",
    "size": 960,
    "blocks": 4,
    "replication_factor": 2
  }
]

=== Bloki na DataNodes ===
  dn-0: 3 bloków
    przykład: [{'block_id': 'demo.txt::block::0', 'size_bytes': 256}, {'block_id': 'demo.txt::block::2', 'size_bytes': 256}]
  dn-1: 3 bloków
    przykład: [{'block_id': 'demo.txt::block::0', 'size_bytes': 256}, {'block_id': 'demo.txt::block::1', 'size_bytes': 256}]
  dn-2: 2 bloków
    przykład: [{'block_id': 'demo.txt::block::1', 'size_bytes': 256}, {'block_id': 'demo.txt::block::2', 'size_bytes': 256}]


## Test 3 - update jednego znaku (tylko zmienione bloki)

Przy `BLOCK_SIZE=256` znak na pozycji 300 leży w **bloku 1**.  
Oczekujemy `changed_block_count == 1` - **bez** kasowania całego artefaktu.



In [6]:
ONE_CHAR_ART = "one_char.txt"
base = "A" * 300 + "X" + "B" * 300
client.create(ONE_CHAR_ART, base, replication_factor=2)

modified = base[:300] + "Y" + base[301:]
result = client.update(ONE_CHAR_ART, modified)

assert client.get(ONE_CHAR_ART) == modified
assert result["changed_block_count"] == 1, result

print("OK - update 1 znaku")
print("  zmienione bloki:", result["changed_block_count"])
print("  bez zmian (block_id):", len(result["unchanged_block_ids"]))



OK - update 1 znaku
  zmienione bloki: 1
  bez zmian (block_id): 2


## Test 4 - delete

Usuwamy metadane w NameNode i bloki na wszystkich replikach.



In [7]:
client.delete(ARTIFACT)

artifacts = ray.get(name_node.list_artifacts.remote())
assert not any(a["name"] == ARTIFACT for a in artifacts), "artefakt nadal w metadanych"

for node_id, blocks in client.list_all_blocks_on_nodes().items():
    if isinstance(blocks, list):
        assert not any(ARTIFACT in b["block_id"] for b in blocks), node_id

print("OK - delete")
print(json.dumps(client.list_cluster_state(), indent=2, ensure_ascii=False))



OK - delete
{
  "artifacts": [
    {
      "name": "one_char.txt",
      "size": 601,
      "blocks": 3,
      "replication_factor": 2
    }
  ],
  "data_nodes": [
    {
      "node_id": "dn-0",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-0",
        "status": "ALIVE",
        "block_count": 2
      }
    },
    {
      "node_id": "dn-1",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-1",
        "status": "ALIVE",
        "block_count": 2
      }
    },
    {
      "node_id": "dn-2",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-2",
        "status": "ALIVE",
        "block_count": 2
      }
    }
  ]
}


## Test 5 - awaria DataNode + replikacja

Tworzymy artefakt z RF=2, zgłaszamy awarię `dn-1`. NameNode kopiuje bloki na żywy węzeł.  
`get` musi zwrócić tę samą treść co przed awarią.



In [8]:
FAIL_ART = "failover.txt"
fail_content = "Z" * 600
client.create(FAIL_ART, fail_content, replication_factor=2)

print("Przed awarią:")
print(json.dumps(client.list_cluster_state(), indent=2, ensure_ascii=False))

rep = client.report_node_failure("dn-1")
assert client.get(FAIL_ART) == fail_content
assert len(rep.get("replications", [])) > 0, "brak akcji replikacji"

state = client.list_cluster_state()
dn1 = next(n for n in state["data_nodes"] if n["node_id"] == "dn-1")
assert dn1["status"] == "DEAD"

print("\nOK - failover, replikacje:", len(rep["replications"]))
print(json.dumps(state, indent=2, ensure_ascii=False))



Przed awarią:
{
  "artifacts": [
    {
      "name": "failover.txt",
      "size": 600,
      "blocks": 3,
      "replication_factor": 2
    },
    {
      "name": "one_char.txt",
      "size": 601,
      "blocks": 3,
      "replication_factor": 2
    }
  ],
  "data_nodes": [
    {
      "node_id": "dn-0",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-0",
        "status": "ALIVE",
        "block_count": 4
      }
    },
    {
      "node_id": "dn-1",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-1",
        "status": "ALIVE",
        "block_count": 4
      }
    },
    {
      "node_id": "dn-2",
      "status": "ALIVE",
      "remote": {
        "node_id": "dn-2",
        "status": "ALIVE",
        "block_count": 4
      }
    }
  ]
}

OK - failover, replikacje: 4
{
  "artifacts": [
    {
      "name": "failover.txt",
      "size": 600,
      "blocks": 3,
      "replication_factor": 2
    },
    {
      "name": "one_char.txt",
      "size": 601,

## Bonus Ray API (spoza labu)

Na zajęciach były m.in.: podstawowe `@ray.remote`, **named actors**, `max_concurrency`, przekazywanie handle do tasków.

**W tym projekcie (plik `hdfs/data_node.py`):**

| Mechanizm | Po co |
|-----------|--------|
| `scheduling_strategy="SPREAD"` | DataNodes na różnych workerach klastra |
| `max_restarts=1`, `max_task_retries=1` | odporność na crash procesu aktora |
| `num_cpus=0.25` | jawne wymagania zasobów aktora |

Opcjonalnie można dodać **detached** NameNode:  
`NameNode.options(name="hdfs-nn-ID", lifetime="detached").remote()` + `ray.get_actor(...)`.



In [9]:
from pathlib import Path
import hdfs.data_node as dn_mod

text = Path(dn_mod.__file__).read_text(encoding="utf-8")
assert "SPREAD" in text, "brak SPREAD w DataNode"
assert "max_restarts" in text, "brak max_restarts w DataNode"
print("OK - bonus API zweryfikowany w hdfs/data_node.py")



OK - bonus API zweryfikowany w hdfs/data_node.py


## Rozproszenie - weryfikacja klastra


In [10]:
import subprocess

try:
    proc = subprocess.run(
        ["ray", "status"],
        capture_output=True,
        text=True,
        timeout=20,
    )
    print(proc.stdout or proc.stderr)
except FileNotFoundError:
    print("ray CLI niedostępne - nodes():")
    for n in ray.nodes():
        print(" ", n.get("NodeID", n)[:16], "alive=", n.get("Alive"))
    print("Resources:", ray.cluster_resources())



======== Autoscaler status: 2026-06-01 02:52:22.014987 ========
Node status
---------------------------------------------------------------
Active:
 1 node_e96b4cc4fd1840185e8ffc53351ee75d9e3bedadd37d8881d75d587e
Pending:
 (no pending nodes)
Recent failures:
 (no failures)

Resources
---------------------------------------------------------------
Total Usage:
 0.75/8.0 CPU
 0B/5.14GiB memory
 0B/2.20GiB object_store_memory

From request_resources:
 (none)
Pending Demands:
 (no resource demands)



In [11]:
ray.shutdown()
print("Ray shutdown.")



Ray shutdown.
